# Chapter 12. Deep Computer Vision using Convolution Neural Networks

This chapter focuses on Convolution Neural Networks (CNNs) which have been around since the 80s and used to research and develop perception for computers. Just as we process our world vidually with specialized intution of features, CNNs attempt to as well. They are also applied to other domains such as audio and natural language processing.

### The visual cortex

Studies of the architecture of real neurons in the visual cortex found a design that involved local neurons keying into specific patterns in small regions of teh visual field called 'receptive fields'; this was fed upward to neurons with larger receptive fields. Sciencists believed that the inputs to these higher-level neurons was the ouptut of the lower-level neurons and allows the visual cortex to identify complex patterns that are derived from the small ones in lower levels.

A groundbreaking 1998 paper by Yann LeCun introduced the _LeNet-5_ architecture, which was used by banks to recognize handwritten digits on checks.


The key elements this model architecture introduced for CNNs was the building blocks of _convolutional layers_ and _pooling layers_.

### Convolutional Layers
The most important building block in which neurons are connected to specific receptive fields, not every single pixel and following a hierarchical structure that allows them to concentrate small low-level features in one layer, and assemble them into larger higher-level features in the next.

- _zero padding_ : Adding zeroes around inputs in order for a layer to have the same height and width as the previous layer
- _stride_ : The horizontal or vertical step size from one receptive field to the next - helpful for reducing computational complexity when feeding a large input layer to a smaller layer

#### Filters
A neuron's weights can be represented as a small image the size of the receptive field. These are called _filters_ or _convolution kernels_.

Using this technique neurons can focus on particular aspects of an image by ignoring or filtering out everything else, this contrasts CNNs from other deep learning methods where the goal is to divide the system into a sequence of specialized modules. A layer full of neurons using the same filter outputs a _feature map_, which highlights the areas in an image that activate the filter the most.

A convolutional layer simultaneously applies multiple trainable filters to its inputs, making it capable of detecting multiple features anywhere in its inputs. What this meens is that the feature maps chare parameters which redcuces the number of parameter in the model, so once the CNN has leanred to recognize a pattern in one location, it can recognize it in any other location.

#### Implementing Convolutional Layers with PyTorch

Loading sample images represented as a Python list of NumPy unsigned byte arrays, so stack these images into a single NumPy array, then convert it to a 32-bit floating point tensor, and rescale the pixel values from 0-255 to 0-1:

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images

sample_images = np.stack(load_sample_images()["images"])  # why do we stack?
print(f"Loaded {len(sample_images)} images, of shape {sample_images[0].shape}")
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

Loaded 2 images, of shape (427, 640, 3)


In [2]:
print(sample_images.shape)

torch.Size([2, 427, 640, 3])


The numbers represent the number of images (2), height (427), width (640) in pixesl, and the number of color channels (3) representing red, green, and blue. PyTorch expects the channel dimension before height and width, so we need to permute the dimensions using the `permute()` method:

In [3]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)  # pyright: ignore[reportAttributeAccessIssue]
print(sample_images_permuted.shape)

torch.Size([2, 3, 427, 640])


We'll also add a center crop

In [4]:
import torchvision
import torchvision.transforms.v2 as T

cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
print(cropped_images.shape)

torch.Size([2, 3, 70, 120])


PyTorch provides teh `nn.Conv2d` layer for creating convolutional layers, here we'll create one with 32 filters each of size 7x7 `kernel_size=7` and apply it to our two images.

In [6]:
import torch.nn as nn  # noqa: PLR0402

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
feature_maps = conv_layer(cropped_images)
print(feature_maps.shape)

torch.Size([2, 32, 64, 114])


The model got 32 output channels as we specified, from the original 3, but it also lost some size losing 6 pixels in each dimension (length and width). This is due to the default padding being 0 or _valid_ with the function we used. Setting `padding="same"` will keep things .. the same.

In [7]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")

In [9]:
feature_maps = conv_layer(cropped_images)
print(feature_maps.shape)

torch.Size([2, 32, 70, 120])


An `nn.Conv2d` layer holds all the layer's parameters, including the kernels and biases, _weight_ and _bias_ attributes.

In [11]:
print(f"weight: {conv_layer.weight.shape}")
print(f"bias: {conv_layer.bias.shape}")  # type: ignore

weight: torch.Size([32, 3, 7, 7])
bias: torch.Size([32])
